In [1]:

# This model predicts user proficiency level (A1–C2)
# based on placement test quiz performance

import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder


In [2]:
# -----------------------------
# LOAD DATASET
# -----------------------------
# Expected: one row per user placement test

df = pd.read_json("Data.json")

# -----------------------------
# FEATURE SELECTION
# -----------------------------
# Later come dynamically from backend/frontend logs

FEATURES = [
    "total_questions",
    "correct_answers",
    "accuracy",
    "easy_accuracy",
    "medium_accuracy",
    "hard_accuracy",
    "avg_time_per_question",

]
TARGET = "expertise_level"  # A1, A2, B1, B2, C1, C2

X = df[FEATURES]
y = df[TARGET]


FileNotFoundError: File Data.json does not exist

In [ ]:

# -----------------------------
# LABEL ENCODING
# -----------------------------

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Mapping check 
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print("Label Mapping:", label_mapping)

# -----------------------------
# TRAIN–TEST SPLIT
# -----------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

# -----------------------------
# MODEL DEFINITION (FINAL)
# -----------------------------

model = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="TotalF1",
    iterations=600, #800
    depth=6, #8
    learning_rate=0.05,
    l2_leaf_reg=5,
    class_weights=[1.3, 1.2, 1.0, 1.0, 1.2, 1.4],
    random_seed=42,
    verbose=100
)

# -----------------------------
# MODEL TRAINING
# -----------------------------

model.fit(X_train, y_train)

# -----------------------------
# EVALUATION (testko lagi)
# -----------------------------

y_pred = model.predict(X_test)

print("\nMacro F1 Score:", f1_score(y_test, y_pred, average="macro"))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

# -----------------------------
# SAFETY LAYER (POST-PROCESSING)
# -----------------------------

def apply_safety_rules(predicted_level, features):
    """
    predicted_level: string (A1–C2)
    features: dict of user features
    """
    if predicted_level == "C2" and features["hard_accuracy"] < 0.75:
        return "C1"

    if predicted_level == "A1" and features["accuracy"] > 0.55:
        return "A2"

    return predicted_level

# -----------------------------
# SINGLE USER PREDICTION (API STYLE)
# -----------------------------

def predict_user_level(user_features: dict):
    """
    user_features: dictionary from backend after placement test
    """
    input_df = pd.DataFrame([user_features])
    encoded_pred = model.predict(input_df)[0]
    predicted_level = label_encoder.inverse_transform([int(encoded_pred)])[0]

    final_level = apply_safety_rules(predicted_level, user_features)
    return final_level


user = {
    "total_questions": 40,
    "correct_answers": 30,
    "accuracy": 0.75,
    "easy_accuracy": 0.9,
    "medium_accuracy": 0.7,
    "hard_accuracy": 0.6,
    "avg_time_per_question": 6.8,
  
}

print("Predicted Level:", predict_user_level(user))


